# Silver og Gold: SQL-transformasjoner

Bygger Silver- og Gold-lagene som SQL-tabeller i Unity Catalog.

**Forutsetning:** Bronze-tabellene er allerede fylt av `01_bronze_ingest`.

**Denne notebooken er 100 % SQL** — ingen Python.

## Opprett schemas

In [0]:
-- Opprett schemas for Silver og Gold
CREATE SCHEMA IF NOT EXISTS pensjon_lakehouse.silver;
CREATE SCHEMA IF NOT EXISTS pensjon_lakehouse.gold;

## Silver

Silver-laget renser og kobler Bronze-data. Tre tabeller:

1. **befolkning_pensjon** — befolkning per kommune og år, med andel 55+
2. **befolkning_aldersgrupper** — befolkning fordelt på 8 aldersgrupper
3. **naering_pensjon** — næringer med lønnstakere, månedslønn og estimert pensjonsvolum

### Silver: befolkning_pensjon

Filtrerer bort fylker (beholder kun 4-sifrede kommunekoder), fjerner null-verdier og ikke-numeriske aldre, og beregner andel 55+ per kommune per år.

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.silver.befolkning_pensjon AS
SELECT
    Region_code  AS kommune_code,
    Region_label AS kommune_label,
    Tid_code     AS year,
    SUM(value)   AS total_befolkning,
    SUM(
        CASE
            WHEN CAST(REGEXP_EXTRACT(Alder_label, '^(\\d+)', 1) AS INT) >= 55
            THEN value
            ELSE 0
        END
    ) AS pension_age_befolkning,
    ROUND(
        SUM(
            CASE
                WHEN CAST(REGEXP_EXTRACT(Alder_label, '^(\\d+)', 1) AS INT) >= 55
                THEN value
                ELSE 0
            END
        ) / NULLIF(SUM(value), 0),
        4
    ) AS pension_age_share,
    current_timestamp() AS _cleaned_ts
FROM pensjon_lakehouse.bronze.ssb_befolkning_raw
WHERE LENGTH(Region_code) = 4
  AND value IS NOT NULL
  AND REGEXP_EXTRACT(Alder_label, '^(\\d+)', 1) != ''
GROUP BY Region_code, Region_label, Tid_code
HAVING SUM(value) > 0
ORDER BY Tid_code, pension_age_share DESC

In [0]:
SELECT * FROM pensjon_lakehouse.silver.befolkning_pensjon LIMIT 10

### Silver: befolkning_aldersgrupper

Grupperer enkeltalder i 8 pensjonsrelevante aldersgrupper: 0-19, 20-34, 35-49, 50-54, 55-61, 62-66, 67-74, 75+.

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.silver.befolkning_aldersgrupper AS
WITH alder AS (
    SELECT
        Region_code  AS kommune_code,
        Region_label AS kommune_label,
        Tid_code     AS year,
        CAST(REGEXP_EXTRACT(Alder_label, '^(\\d+)', 1) AS INT) AS alder,
        value AS antall
    FROM pensjon_lakehouse.bronze.ssb_befolkning_raw
    WHERE LENGTH(Region_code) = 4
      AND value IS NOT NULL
      AND REGEXP_EXTRACT(Alder_label, '^(\\d+)', 1) != ''
),

grouped AS (
    SELECT
        kommune_code,
        kommune_label,
        year,
        CASE
            WHEN alder BETWEEN  0 AND 19 THEN '0-19'
            WHEN alder BETWEEN 20 AND 34 THEN '20-34'
            WHEN alder BETWEEN 35 AND 49 THEN '35-49'
            WHEN alder BETWEEN 50 AND 54 THEN '50-54'
            WHEN alder BETWEEN 55 AND 61 THEN '55-61'
            WHEN alder BETWEEN 62 AND 66 THEN '62-66'
            WHEN alder BETWEEN 67 AND 74 THEN '67-74'
            ELSE '75+'
        END AS aldersgruppe,
        CASE
            WHEN alder BETWEEN  0 AND 19 THEN 1
            WHEN alder BETWEEN 20 AND 34 THEN 2
            WHEN alder BETWEEN 35 AND 49 THEN 3
            WHEN alder BETWEEN 50 AND 54 THEN 4
            WHEN alder BETWEEN 55 AND 61 THEN 5
            WHEN alder BETWEEN 62 AND 66 THEN 6
            WHEN alder BETWEEN 67 AND 74 THEN 7
            ELSE 8
        END AS aldersgruppe_sortering,
        SUM(antall) AS befolkning
    FROM alder
    GROUP BY kommune_code, kommune_label, year, aldersgruppe, aldersgruppe_sortering
)

SELECT
    *,
    ROUND(
        CAST(befolkning AS DOUBLE)
        / NULLIF(SUM(befolkning) OVER (PARTITION BY kommune_code, year), 0),
        4
    ) AS aldersgruppe_andel,
    current_timestamp() AS _cleaned_ts
FROM grouped
ORDER BY year, kommune_label, aldersgruppe_sortering

In [0]:
SELECT * FROM pensjon_lakehouse.silver.befolkning_aldersgrupper LIMIT 10

### Silver: naering_pensjon

Pivoterer lønnstakere og månedslønn til kolonner, og beregner estimert pensjonsvolum: `lønnstakere × månedslønn × 12 × 2 % OTP`.

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.silver.naering_pensjon AS
WITH pivoted AS (
    SELECT
        NACE2007_code  AS naering_code,
        NACE2007_label AS naering_label,
        Tid_code       AS kvartal,
        MAX(CASE WHEN ContentsCode_code = 'Lonsstakere' THEN value END) AS lonsstakere,
        MAX(CASE WHEN ContentsCode_code = 'GjMdTotal'   THEN value END) AS manedslonn
    FROM pensjon_lakehouse.bronze.ssb_lonn_sysselsetting_raw
    WHERE value IS NOT NULL
    GROUP BY NACE2007_code, NACE2007_label, Tid_code
)

SELECT
    naering_code,
    naering_label,
    kvartal,
    CAST(lonsstakere AS INT) AS lonsstakere,
    CAST(manedslonn AS INT)  AS manedslonn,
    ROUND(lonsstakere * COALESCE(manedslonn, 0) * 12 * 0.02) AS estimert_pensjonsvolum,
    current_timestamp() AS _cleaned_ts
FROM pivoted
WHERE lonsstakere IS NOT NULL
ORDER BY estimert_pensjonsvolum DESC NULLS LAST

In [0]:
SELECT * FROM pensjon_lakehouse.silver.naering_pensjon LIMIT 10

## Gold

Gold-laget bygger analyseklare tabeller fra Silver. Fem tabeller:

1. **pensjonsandel_trend** — vektet landsgjennomsnitt per år
2. **top_kommuner_pensjonsalder** — topp 20 kommuner med høyest andel 55+
3. **naering_pensjonsvolum** — næringer rangert etter pensjonsvolum
4. **aldersgruppe_fordeling** — fordeling siste år
5. **aldersgruppe_trend** — andeler per aldersgruppe over tid

### Gold: pensjonsandel_trend

Vektet nasjonalt gjennomsnitt: `SUM(55+) / SUM(total)`, ikke `AVG(kommuneandeler)`. Et uvektet snitt ville gitt feil bilde fordi små kommuner ville veid like mye som store.

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.pensjonsandel_trend AS
SELECT
    year,
    SUM(pension_age_befolkning) AS total_55_pluss,
    SUM(total_befolkning)       AS total_befolkning,
    ROUND(
        CAST(SUM(pension_age_befolkning) AS DOUBLE)
        / NULLIF(SUM(total_befolkning), 0)
        * 100,
        2
    ) AS pensjonsandel_pst
FROM pensjon_lakehouse.silver.befolkning_pensjon
GROUP BY year
ORDER BY year

In [0]:
SELECT * FROM pensjon_lakehouse.gold.pensjonsandel_trend

### Gold: top_kommuner_pensjonsalder

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.top_kommuner_pensjonsalder AS
SELECT
    kommune_code,
    kommune_label,
    year,
    total_befolkning,
    pension_age_befolkning,
    pension_age_share
FROM pensjon_lakehouse.silver.befolkning_pensjon
WHERE year = (SELECT MAX(year) FROM pensjon_lakehouse.silver.befolkning_pensjon)
ORDER BY pension_age_share DESC
LIMIT 20

In [0]:
SELECT * FROM pensjon_lakehouse.gold.top_kommuner_pensjonsalder

### Gold: naering_pensjonsvolum

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.naering_pensjonsvolum AS
SELECT
    naering_code,
    naering_label,
    kvartal,
    lonsstakere,
    manedslonn,
    estimert_pensjonsvolum
FROM pensjon_lakehouse.silver.naering_pensjon
WHERE kvartal = (SELECT MAX(kvartal) FROM pensjon_lakehouse.silver.naering_pensjon)
  AND estimert_pensjonsvolum IS NOT NULL
ORDER BY estimert_pensjonsvolum DESC

In [0]:
SELECT * FROM pensjon_lakehouse.gold.naering_pensjonsvolum

### Gold: aldersgruppe_fordeling (siste år)

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.aldersgruppe_fordeling AS
SELECT
    aldersgruppe,
    aldersgruppe_sortering,
    SUM(befolkning) AS befolkning,
    ROUND(
        CAST(SUM(befolkning) AS DOUBLE)
        / NULLIF(SUM(SUM(befolkning)) OVER (), 0),
        4
    ) AS andel
FROM pensjon_lakehouse.silver.befolkning_aldersgrupper
WHERE year = (SELECT MAX(year) FROM pensjon_lakehouse.silver.befolkning_aldersgrupper)
GROUP BY aldersgruppe, aldersgruppe_sortering
ORDER BY aldersgruppe_sortering

In [0]:
SELECT * FROM pensjon_lakehouse.gold.aldersgruppe_fordeling

### Gold: aldersgruppe_trend

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.aldersgruppe_trend AS
SELECT
    year,
    aldersgruppe,
    aldersgruppe_sortering,
    SUM(befolkning) AS befolkning,
    ROUND(
        CAST(SUM(befolkning) AS DOUBLE)
        / NULLIF(SUM(SUM(befolkning)) OVER (PARTITION BY year), 0),
        4
    ) AS andel
FROM pensjon_lakehouse.silver.befolkning_aldersgrupper
GROUP BY year, aldersgruppe, aldersgruppe_sortering
ORDER BY year, aldersgruppe_sortering

In [0]:
SELECT * FROM pensjon_lakehouse.gold.aldersgruppe_trend

## Ferdig

Alle tabeller ligger nå i Unity Catalog:

```
pensjon_lakehouse.bronze.ssb_befolkning_raw
pensjon_lakehouse.bronze.ssb_lonn_sysselsetting_raw
pensjon_lakehouse.silver.befolkning_pensjon
pensjon_lakehouse.silver.befolkning_aldersgrupper
pensjon_lakehouse.silver.naering_pensjon
pensjon_lakehouse.gold.pensjonsandel_trend
pensjon_lakehouse.gold.top_kommuner_pensjonsalder
pensjon_lakehouse.gold.naering_pensjonsvolum
pensjon_lakehouse.gold.aldersgruppe_fordeling
pensjon_lakehouse.gold.aldersgruppe_trend
```

Du kan nå bruke disse direkte i Databricks SQL Dashboard eller koble Power BI mot Databricks SQL Warehouse.